# Build Gold Mart with Spark and Iceberg

Этот notebook читает `gold`-витрины напрямую из Iceberg через Spark и создает новую витрину в каталоге `gold`.

## 1. Imports and SparkSession

Spark подключается к `spark-connect`, а Iceberg catalog `gold` настраивается на базе конфигов из `spark/conf/spark-defaults.conf` и `trino-config/catalog/dlh-gold.properties`.

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
conf = (
    pyspark.SparkConf()
        # 2. Настройка самого каталога Iceberg (JDBC + Postgres) через conf.set
        .set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .set("spark.sql.catalog.gold", "org.apache.iceberg.spark.SparkCatalog")
        .set("spark.sql.catalog.gold.type", "jdbc")
        .set("spark.sql.catalog.gold.uri", "jdbc:postgresql://postgres:5432/metastore?socketTimeout=10&connectTimeout=5")
        .set("spark.sql.catalog.gold.jdbc.driver-class", "org.postgresql.Driver")
        .set("spark.sql.catalog.gold.jdbc.user", "postgres")
        .set("spark.sql.catalog.gold.jdbc.password", "postgres")
        .set("spark.sql.catalog.gold.catalog-name", "gold")

        # Указываем Iceberg использовать файловый движок Hadoop (чтобы он слушал fs.s3a)
        .set("spark.sql.catalog.gold.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
        .set("spark.sql.catalog.gold.warehouse", "s3a://gold/jdbc")

        # 3. Твой прошлый рабочий конфиг для MinIO (Hadoop fs.s3a)
        .set("fs.s3a.access.key", "minioadmin")
        .set("fs.s3a.secret.key", "minioadmin")
        .set("fs.s3a.endpoint", "http://minio:9002")
        .set("fs.s3a.path.style.access", "true")
        .set("fs.s3a.connection.ssl.enabled", "false")
        .set("fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [3]:
# Инициализируем билдер
spark = SparkSession.builder \
    .appName('BuildGoldMartWithSpark') \
    .config(conf=conf) \
    .remote('sc://spark-connect:15002') \
    .getOrCreate()

/opt/conda/lib/python3.11/site-packages/pyspark/sql/connect/session.py:185: UserWarning: Cannot modify the value of a static config: spark.sql.extensions.
  warnings.warn(str(e))


In [4]:
spark.conf.get("spark.sql.catalog.gold.uri")

'jdbc:postgresql://postgres:5432/metastore?socketTimeout=10&connectTimeout=5'

In [8]:
try:
    # Читаем таблицу напрямую через стандартный JDBC формат Spark
    postgres_df = spark.read \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://postgres:5432/metastore") \
        .option("dbtable", "iceberg_tables") \
        .option("user", "postgres") \
        .option("password", "postgres") \
        .option("driver", "org.postgresql.Driver") \
        .load() \
        .filter("catalog_name = 'gold'")

    # Если подключение успешно, выведем схему и количество строк
    print("Успех! Сеть и драйвер работают корректно.")
    print(f"Количество записей в iceberg_tables: {postgres_df.count()}")
    postgres_df.show(1, truncate=True)

except Exception as e:
    print("\n[ОШИБКА] Прямое подключение тоже не удалось!")
    print("Детали ошибки:")
    print(e)

Успех! Сеть и драйвер работают корректно.
Количество записей в iceberg_tables: 6
+------------+---------------+--------------------+--------------------+--------------------------+------------+
|catalog_name|table_namespace|          table_name|   metadata_location|previous_metadata_location|iceberg_type|
+------------+---------------+--------------------+--------------------+--------------------------+------------+
|        gold|           tpch|gold_dim_part_sup...|s3://gold/jdbc/tp...|      s3://gold/jdbc/tp...|       TABLE|
+------------+---------------+--------------------+--------------------+--------------------------+------------+
only showing top 1 row



## 2. Read gold fact data from Iceberg

In [11]:
source_table = 'gold.tpch.gold_fct_orders_items'

gold_fact = spark.table(source_table)

print(f'Rows: {gold_fact.count()}')
gold_fact.printSchema()
# gold_fact.show(1, truncate=True)

Rows: 60175
root
 |-- order_item_key: string (nullable = true)
 |-- order_key: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- order_status_code: string (nullable = true)
 |-- part_key: long (nullable = true)
 |-- supplier_key: long (nullable = true)
 |-- return_status_code: string (nullable = true)
 |-- order_line_number: integer (nullable = true)
 |-- order_line_status_code: string (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- commit_date: date (nullable = true)
 |-- receipt_date: date (nullable = true)
 |-- ship_mode_name: string (nullable = true)
 |-- supplier_cost_amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_price: double (nullable = true)
 |-- tax_rate: double (nullable = true)
 |-- order_item_count: integer (nullable = true)
 |-- quantity: double (nullable = true)
 |-- gross_item_sales_amount: double (nulla

## 3. Build a new mart

In [13]:
order_sales_mart = (
    gold_fact
    .withColumn('ship_month', F.date_trunc('month', F.col('ship_date')))
    .groupBy('ship_month', 'return_status_code', 'order_line_status_code')
    .agg(
        F.count('*').alias('line_item_count'),
        F.sum('quantity').alias('total_quantity'),
        F.sum('gross_item_sales_amount').alias('gross_item_sales_amount'),
        F.sum('discounted_item_sales_amount').alias('discounted_item_sales_amount'),
        F.sum('net_item_sales_amount').alias('net_item_sales_amount'),
        F.avg('base_price').alias('avg_base_price'),
        F.avg('discount_percentage').alias('avg_discount_percentage')
    )
)

order_sales_mart.show(10, truncate=True)

+-------------------+------------------+----------------------+---------------+--------------+-----------------------+----------------------------+---------------------+------------------+-----------------------+
|         ship_month|return_status_code|order_line_status_code|line_item_count|total_quantity|gross_item_sales_amount|discounted_item_sales_amount|net_item_sales_amount|    avg_base_price|avg_discount_percentage|
+-------------------+------------------+----------------------+---------------+--------------+-----------------------+----------------------------+---------------------+------------------+-----------------------+
|1994-10-01 00:00:00|                 A|                     F|            379|        9701.0|   1.3451959319999993E7|        1.2769729845500005E7| 1.3277841192497002E7|1394.3805540897106|   0.050870712401055354|
|1995-06-01 00:00:00|                 N|                     O|            314|        7968.0|   1.1210983040000007E7|             1.06641652697E7| 

## 4. Write to Iceberg catalog `gold.tpch`

In [14]:
target_table = 'gold.tpch.order_sales_monthly_summary'

(order_sales_mart.writeTo(target_table)
    .createOrReplace())

print(f'Written to Iceberg table: {target_table}')

Written to Iceberg table: gold.tpch.order_sales_monthly_summary


In [15]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS gold.tpch')
print('Namespace gold.tpch is ready')

Namespace gold.tpch is ready


## 5. Verify the new Iceberg table

In [16]:
check_df = spark.table('gold.tpch.order_sales_monthly_summary')
print(f'Rows in Iceberg mart: {check_df.count()}')
check_df.orderBy('ship_month').show(10, truncate=True)

Rows in Iceberg mart: 128
+-------------------+------------------+----------------------+---------------+--------------+-----------------------+----------------------------+---------------------+------------------+-----------------------+
|         ship_month|return_status_code|order_line_status_code|line_item_count|total_quantity|gross_item_sales_amount|discounted_item_sales_amount|net_item_sales_amount|    avg_base_price|avg_discount_percentage|
+-------------------+------------------+----------------------+---------------+--------------+-----------------------+----------------------------+---------------------+------------------+-----------------------+
|1992-01-01 00:00:00|                 R|                     F|             39|        1040.0|     1530408.4800000004|          1442039.7338999999|   1507687.7939249997|1460.9153846153843|   0.052820512820512845|
|1992-01-01 00:00:00|                 A|                     F|             69|        1689.0|             2411863.03|    